# P1_02 · Model parity on GPU (P1-6 acceptance)

1. **MedSAM**: our wrapper vs. the official `MedSAM_Inference.py` (bowang-lab/MedSAM, pinned commit) on 3 Kvasir test images with ground-truth boxes. Required: Dice ≥ 0.99 on every image.
2. **SAM ViT-B and ViT-H**: our wrapper vs. a direct `SamPredictor` call. Required: pixel-identical masks.

Results: `<Drive>/results/parity/`. Needs P1_00 first.

**Before you start** (once per Google account):
1. (Only if the repo is private again) Colab left bar → 🔑 *Secrets* → add `GH_TOKEN` (read access
   to `geekRoy0126/promptshift`), switch on *Notebook access*. Never paste a token into a cell.
2. *Runtime → Change runtime type → T4 GPU*.
3. *Runtime → Run all*. Allow Google Drive access when asked.

All data, weights and results go to `<Drive>/` = `MyDrive/cs5330/cs5330/promptshift/`
(set by `DRIVE_ROOT` in the setup cell), so a disconnect loses nothing:
rerun the notebook and finished steps are skipped.

In [ ]:
# Parameters (edit before Run all if needed)
SAM_MODELS = ["sam_vit_b", "sam_vit_h"]

In [ ]:
# SETUP: identical in every notebook (generated by scripts/build_notebooks.py).
import base64
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/geekRoy0126/promptshift.git"
REF = "main"  # branch, tag or commit to run
# Storage root on Google Drive (data, weights, results, checkpoints). Change it here if needed.
DRIVE_ROOT = "/content/drive/MyDrive/cs5330/cs5330/promptshift"

IN_COLAB = "COLAB_RELEASE_TAG" in os.environ
if IN_COLAB:
    from google.colab import drive, userdata

    drive.mount("/content/drive")
    os.environ["PROMPTSHIFT_ROOT"] = DRIVE_ROOT  # read by promptshift.paths in every script
    Path(DRIVE_ROOT).mkdir(parents=True, exist_ok=True)
    REPO = Path("/content/promptshift")
    try:  # optional: the repo is public; GH_TOKEN is only needed if it is made private again
        _token = userdata.get("GH_TOKEN")
    except Exception:  # secret missing or notebook access not granted
        _token = ""
    _auth = base64.b64encode(f"x-access-token:{_token}".encode()).decode() if _token else ""

    def _git(*args, auth=False):
        pre = ["-c", f"http.extraHeader=Authorization: Basic {_auth}"] if auth and _auth else []
        r = subprocess.run(["git", *pre, "-C", str(REPO), *args], capture_output=True, text=True)
        if r.returncode:
            msg = r.stderr or r.stdout
            for s in (_token, _auth):
                msg = msg.replace(s, "***") if s else msg
            raise RuntimeError(f"git {args[0]} failed: {msg}")

    REPO.mkdir(parents=True, exist_ok=True)
    if not (REPO / ".git").is_dir():
        _git("init", "-q")
    _git("fetch", "-q", "--depth", "1", REPO_URL, REF, auth=True)
    _git("reset", "-q", "--hard", "FETCH_HEAD")  # throwaway clone; all outputs live on Drive
    del _token, _auth
    _pip = [sys.executable, "-m", "pip", "install", "-q", "-e", str(REPO)]
    r = subprocess.run(_pip, capture_output=True, text=True)
    if r.returncode:
        print(r.stdout[-3000:], r.stderr[-3000:])
        raise RuntimeError("pip install -e failed (log above)")
else:  # local CPU run: use the checkout this notebook lives in
    REPO = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())

os.chdir(REPO)
sys.path.insert(0, str(REPO / "src"))
os.environ["PYTHONUNBUFFERED"] = "1"
# scripts must import this checkout's code, whatever else is installed
os.environ["PYTHONPATH"] = os.pathsep.join([str(REPO / "src"), os.environ.get("PYTHONPATH", "")])


def run(*args, check=True):
    """Run a repo script with live output; stop the notebook if it fails. Returns the output."""
    cmd = [sys.executable, *map(str, args)]
    print("$ python", " ".join(cmd[1:]), flush=True)
    p = subprocess.Popen(cmd, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
    out = []
    while chunk := p.stdout.read1(4096):
        text = chunk.decode(errors="replace")
        out.append(text)
        sys.stdout.write(text)
        sys.stdout.flush()
    if p.wait() and check:
        raise RuntimeError(f"exit code {p.returncode}: python {' '.join(cmd[1:])}")
    return "".join(out)


def check_gpu():
    import torch

    if IN_COLAB and not torch.cuda.is_available():
        raise RuntimeError("No GPU. Runtime > Change runtime type > T4 GPU, then Run all again.")
    print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none (CPU)")


def git_out(*args):
    r = subprocess.run(["git", "-C", str(REPO), *args], capture_output=True, text=True)
    return r.stdout.strip()


SHA = git_out("rev-parse", "--short", "HEAD")
print(f"repo {REPO} @ {SHA} (Colab: {IN_COLAB})")

In [ ]:
check_gpu()

## 1. Run the parity checks

In [ ]:
out = run("scripts/model_parity.py", *[a for m in SAM_MODELS for a in ("--sam", m)],
          check=False)

## 2. Results

In [ ]:
import json

import pandas as pd
from IPython.display import display

from promptshift.paths import get_paths

parity = get_paths().results / "parity"
medsam = pd.read_csv(parity / "medsam_parity.csv")
sam = pd.read_csv(parity / "sam_parity.csv")
display(medsam)
display(sam)
print(json.loads((parity / "environment.json").read_text()))
assert "MedSAM parity: PASS" in out and "SAM parity: PASS" in out, "see run output"
assert medsam["pass"].all(), "MedSAM parity FAILED (Dice < 0.99)"
assert sam["identical"].all(), "SAM wrapper differs from SamPredictor"
assert set(SAM_MODELS) <= set(sam["model"]), "a SAM variant was not checked"
print("\nParity PASS. P1_02 done. Next: P1_03_kvasir_box_pilot.")